In [1]:
# Load dataset and inspect the first 5 rows
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Read dataset
df = pd.read_csv(r'C:\Users\theha\OneDrive\Desktop\Projects\2. State Employee Credit Card Transaction\Data_raw\State_Employee_Credit_Card_Transactions_Cleaned.csv', low_memory=False)

# Screening of data structure
df.head()

,FISCAL_YEAR,FISCAL_PERIOD,DEPT_NAME,DIV_NAME,MERCHANT,CAT_DESCR,TRANS_DT,MERCHANDISE_AMT,IS_REFUND,AMOUNT_ABS,...,IS_EXACT_DUP,TX_DATE,TX_YEAR,TX_MONTH,DAY_OF_WEEK,IS_WEEKEND,IS_UNKNOWN_DEPT_NAME,IS_UNKNOWN_DIV_NAME,IS_UNKNOWN_MERCHANT,IS_UNKNOWN_CAT_DESCR
0,2025,12,Freire Charter School,Freire Charter School,AMAZON MKTPL*N64SF7SY0,Book Stores,2025-06-06,299.70,0,299.70,...,0,2025-06-06,2025,6,Friday,0,0,0,0,0
1,2025,12,Freire Charter School,Freire Charter School,CITY OF WILMINGTON,Utlts-Elctrc Gas Heating Oil Sanitary Water,2025-06-13,792.48,0,792.48,...,0,2025-06-13,2025,6,Friday,0,0,0,0,0
2,2025,12,Freire Charter School,Freire Charter School,BJS WHOLESALE #0354,Wholesale Clubs,2025-05-27,55.94,0,55.94,...,0,2025-05-27,2025,5,Tuesday,0,0,0,0,0
3,2025,12,Freire Charter School,Freire Charter School,CHICK-FIL-A #05288,Fast Food Restaurants,2025-06-04,549.00,0,549.00,...,0,2025-06-04,2025,6,Wednesday,0,0,0,0,0
4,2025,12,Freire Charter School,Freire Charter School,DISCOUNTMUGS.COM,Miscellaneous General Merchandise,2025-05-30,3392.60,0,3392.60,...,0,2025-05-30,2025,5,Friday,0,0,0,0,0


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 742191 entries, 0 to 742190
Data columns (total 22 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   FISCAL_YEAR           742191 non-null  int64  
 1   FISCAL_PERIOD         742191 non-null  int64  
 2   DEPT_NAME             742191 non-null  object 
 3   DIV_NAME              742191 non-null  object 
 4   MERCHANT              742191 non-null  object 
 5   CAT_DESCR             742191 non-null  object 
 6   TRANS_DT              742191 non-null  object 
 7   MERCHANDISE_AMT       742191 non-null  float64
 8   IS_REFUND             742191 non-null  int64  
 9   AMOUNT_ABS            742191 non-null  float64
 10  IS_HIGH_VALUE         742191 non-null  int64  
 11  KEY_GROUP_SIZE        742191 non-null  int64  
 12  IS_EXACT_DUP          742191 non-null  int64  
 13  TX_DATE               742191 non-null  object 
 14  TX_YEAR               742191 non-null  int64  
 15  

In [3]:
import numpy as np

# Configure column names for dataset
AMOUNT_COL   = "MERCHANDISE_AMT"
DATE_COL     = "TRANS_DT"
MERCHANT_COL = "MERCHANT"
CAT_COL      = "CAT_DESCR"
DIV_COL      = "DIV_NAME"
DEPT_COL     = "DEPT_NAME"

df2 = df.copy()

In [4]:
# Amount & log-transform
df2["amount"] = pd.to_numeric(df2[AMOUNT_COL], errors="coerce").abs()
df2["amount"] = df2["amount"].clip(lower=0)
df2["log_amount"] = np.log1p(df2["amount"])

In [5]:
# Basic time
df2["trans_dt"]   = pd.to_datetime(df2[DATE_COL], errors="coerce")
df2["tx_year"]    = df2["trans_dt"].dt.year
df2["tx_month"]   = df2["trans_dt"].dt.month
df2["day_of_week"]= df2["trans_dt"].dt.dayofweek  # 0=Mon .. 6=Sun
df2["is_weekend"] = df2["day_of_week"].isin([5,6]).astype(int)

In [6]:
df2["TRANS_DT"] = pd.to_datetime(df2["TRANS_DT"], errors="coerce")
df2["TX_YEAR"] = pd.to_numeric(df2["TX_YEAR"], errors="coerce")
df2["TX_MONTH"] = pd.to_numeric(df2["TX_MONTH"], errors="coerce")
df2["DAY_OF_WEEK"] = pd.to_numeric(df2["DAY_OF_WEEK"], errors="coerce")
df2["IS_WEEKEND"] = df2["IS_WEEKEND"].astype(int)

time_features = ["TX_YEAR", "TX_MONTH", "DAY_OF_WEEK", "IS_WEEKEND"]

In [7]:
# Merchant/category rarity
def rarity_by(colname):
    freq = df2[colname].value_counts(normalize=True)  # frequency
    r = (-np.log(freq + 1e-12))                       # change log
    r = (r - r.min()) / (r.max() - r.min() + 1e-12)   # min-max 0..1
    return df2[colname].map(r)                        # map by each row

df2["merchant_rarity"] = rarity_by(MERCHANT_COL)
df2["category_rarity"] = rarity_by(CAT_COL)

In [8]:
# Compared to unit/department habits (z-score by group)
def group_zscore(s, grp):
    g_mean = s.groupby(grp).transform("mean")
    g_std  = s.groupby(grp).transform("std")
    z = (s - g_mean) / g_std
    z = z.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return z

# Z-score by Division (can be thought of as "employee/unit")
df2["z_amount_div"]  = group_zscore(df2["amount"], df2[DIV_COL])

# Z-score by Department
df2["z_amount_dept"] = group_zscore(df2["amount"], df2[DEPT_COL])

In [9]:
# Select the basic feature set to use for the next step
basic_feature_cols = [
    "amount", "log_amount",
    "tx_month", "day_of_week", "is_weekend",
    "merchant_rarity", "category_rarity",
    "z_amount_div", "z_amount_dept"
]

features = df2[basic_feature_cols].copy()
features.head()

,amount,log_amount,tx_month,day_of_week,is_weekend,merchant_rarity,category_rarity,z_amount_div,z_amount_dept
0,299.70,5.706113,6,4,0,1.000000,0.000000,-0.196569,-0.196569
1,792.48,6.676428,6,4,0,0.189212,0.161915,0.159505,0.159505
2,55.94,4.041998,5,1,0,0.352292,0.206554,-0.372705,-0.372705
3,549.00,6.309918,6,2,0,0.712932,0.120388,-0.016429,-0.016429
4,3392.60,8.129647,5,4,0,0.548915,0.288955,2.038304,2.038304


In [10]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Select numeric column
num_cols = features.select_dtypes(include="number").columns.tolist()
X_num = features[num_cols].astype("float64").copy()   # dùng cho IQR/Z-score

# Impute NaN (median, entry-level & robust)
imputer = SimpleImputer(strategy="median")
X_imputed = imputer.fit_transform(X_num)

# Scale for LOF/IF
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

# Convert to DataFrame for easy viewing/recording of indexes
X_imputed_df = pd.DataFrame(X_imputed, columns=num_cols, index=features.index)
X_scaled_df  = pd.DataFrame(X_scaled,  columns=num_cols, index=features.index)

X_num.shape, X_imputed_df.isna().sum().sum(), X_scaled_df.describe().T.head()

((742191, 9),
 np.int64(0),
                 count          mean       std       min       25%       50%  \
 amount       742191.0 -1.378596e-18  1.000001 -0.180945 -0.171139 -0.149585   
 log_amount   742191.0  5.161310e-16  1.000001 -2.677341 -0.740567 -0.065429   
 tx_month     742191.0  9.711443e-17  1.000001 -1.615537 -0.721085 -0.124784   
 day_of_week  742191.0  1.248587e-16  1.000001 -1.455927 -0.853908 -0.251890   
 is_weekend   742191.0 -5.380354e-17  1.000001 -0.353860 -0.353860 -0.353860   
 
                   75%         max  
 amount      -0.067759  148.677329  
 log_amount   0.692215    4.959333  
 tx_month     0.769668    1.664120  
 day_of_week  0.952148    2.156185  
 is_weekend  -0.353860    2.825973  )

In [11]:
col = "log_amount"
x = features[col].dropna()

Q1 = x.quantile(0.25)
Q3 = x.quantile(0.75)
IQR = Q3 - Q1
k = 1.5   # 1.5 = questionable; 3.0 = very unusual

lower = Q1 - k*IQR
upper = Q3 + k*IQR

# Outlier flag
features["iqr_flag"] = ((features[col] < lower) | (features[col] > upper)).astype(int)

# Distance out to ranking
features["iqr_distance"] = 0.0
features.loc[features[col] < lower, "iqr_distance"] = lower - features[col]
features.loc[features[col] > upper, "iqr_distance"] = features[col] - upper

features[["log_amount", "iqr_flag", "iqr_distance"]].head()

,log_amount,iqr_flag,iqr_distance
0,5.706113,0,0.0
1,6.676428,0,0.0
2,4.041998,0,0.0
3,6.309918,0,0.0
4,8.129647,0,0.0


In [12]:
m = features["log_amount"].mean()
s = features["log_amount"].std(ddof=0)

features["zscore"] = (features["log_amount"] - m) / (s + 1e-6)
features["z_flag"] = (features["zscore"].abs() > 3).astype(int)

features[["log_amount", "zscore", "z_flag"]].head()

,log_amount,zscore,z_flag
0,5.706113,0.714656,0
1,6.676428,1.291459,0
2,4.041998,-0.274576,0
3,6.309918,1.073587,0
4,8.129647,2.155324,0


In [13]:
print("IQR flagged:", features["iqr_flag"].sum())
print("Z-score flagged:", features["z_flag"].sum())

IQR flagged: 3445
Z-score flagged: 2472


In [14]:
features.sort_values("iqr_distance", ascending=False).head(10)
features.sort_values("zscore", key=abs, ascending=False).head(10)

,amount,log_amount,tx_month,day_of_week,is_weekend,merchant_rarity,category_rarity,z_amount_div,z_amount_dept,iqr_flag,iqr_distance,zscore,z_flag
303055,379505.58,12.846627,12,6,1,0.386338,0.305173,20.264440,58.913745,1,3.562866,4.959330,1
299836,368335.10,12.816751,12,4,0,0.348968,0.213014,20.922597,108.763067,1,3.532990,4.941570,1
352306,340257.23,12.737460,9,4,0,0.386338,0.305173,18.128551,52.798492,1,3.453699,4.894435,1
480779,217515.00,12.290028,1,4,0,0.409097,0.394419,53.257980,33.674128,1,3.006266,4.628459,1
154649,213034.42,12.269214,8,2,0,0.386338,0.305173,11.205108,32.976013,1,2.985452,4.616086,1
77314,211682.00,12.262845,1,3,0,0.307759,0.266703,55.397974,32.765293,1,2.979084,4.612301,1
632513,207882.20,12.244732,3,3,0,0.571028,0.301692,10.924726,32.173250,1,2.960970,4.601533,1
117258,205228.58,12.231885,11,2,0,0.386338,0.305173,10.780316,31.759791,1,2.948123,4.593896,1
330573,200000.00,12.206078,9,3,0,0.348968,0.213014,11.243839,58.953963,1,2.922316,4.578555,1
240317,191948.45,12.164987,4,2,0,0.386338,0.305173,10.057614,29.690625,1,2.881226,4.554129,1


In [15]:
from sklearn.ensemble import IsolationForest

iso = IsolationForest(
    n_estimators=200,
    contamination=0.01,   # 1% doubt
    random_state=42
)
iso.fit(X_scaled)

# The smaller the score, the more abnormal (reverse the sign to get "if_score_raw" = the abnormal score gets bigger)
features["if_score_raw"] = -iso.score_samples(X_scaled)

In [16]:
from sklearn.neighbors import LocalOutlierFactor

lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=0.01
)
y_pred = lof.fit_predict(X_scaled)   # -1 = outlier, 1 = inlier

features["lof_outlier"] = (y_pred == -1).astype(int)

# Reverse the sign so that the larger the score, the more unusual it is
features["lof_score_raw"] = -lof.negative_outlier_factor_ 

In [17]:
# Select important column
from sklearn.preprocessing import MinMaxScaler

score_cols = ["iqr_distance", "zscore", "if_score_raw", "lof_score_raw"]
score_cols = [c for c in score_cols if c in features.columns]

In [18]:
# Scale to [0,1]
mm = MinMaxScaler()
features[[c+"_01" for c in score_cols]] = mm.fit_transform(features[score_cols])

# Create a composite anomaly score
subcols = [c+"_01" for c in score_cols]
features["anomaly_score"] = features[subcols].mean(axis=1)

# Select Top-k threshold
k = int(0.01 * len(features))  # 1% dữ liệu
features["rank"] = features["anomaly_score"].rank(ascending=False, method="first")
suspicious = features.sort_values("anomaly_score", ascending=False).head(k)

suspicious.head()

,amount,log_amount,tx_month,day_of_week,is_weekend,merchant_rarity,category_rarity,z_amount_div,z_amount_dept,iqr_flag,...,z_flag,if_score_raw,lof_outlier,lof_score_raw,iqr_distance_01,zscore_01,if_score_raw_01,lof_score_raw_01,anomaly_score,rank
303055,379505.58,12.846627,12,6,1,0.386338,0.305173,20.264440,58.913745,1,...,1,0.796655,1,3.410357,1.000000,1.000000,1.000000,2.080413e-10,0.750000,1.0
299836,368335.10,12.816751,12,4,0,0.348968,0.213014,20.922597,108.763067,1,...,1,0.772965,1,3.792239,0.991615,0.997674,0.944020,2.393089e-10,0.733327,2.0
352306,340257.23,12.737460,9,4,0,0.386338,0.305173,18.128551,52.798492,1,...,1,0.771773,0,2.858704,0.969360,0.991502,0.941202,1.628733e-10,0.725516,3.0
480779,217515.00,12.290028,1,4,0,0.409097,0.394419,53.257980,33.674128,1,...,1,0.778220,0,1.910201,0.843778,0.956673,0.956438,8.521197e-11,0.689222,4.0
77314,211682.00,12.262845,1,3,0,0.307759,0.266703,55.397974,32.765293,1,...,1,0.770286,0,1.935588,0.836148,0.954558,0.937688,8.729064e-11,0.682098,5.0


In [19]:
# Export suspect top table
suspicious.to_csv("Top_Suspicious_Transactions.csv", index=False)